In [1]:
from __future__ import annotations

import json
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

# プロジェクトルートを sys.path に追加（notebook/LIST_Transcript/ の2つ上）
_project_root = Path().resolve().parents[1]
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from utils_core.config import ProjectConfig
config = ProjectConfig.from_env()
from research.ca_strategy_poc.scripts import run_buyhold_backtest_bloomberg as bt

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
ROOT = config.data_path.parent  # finance/
BBG_CSV = config.data_path / "Transcript" / "list_port_and_index_price_2015_2026.csv"
BBG_MCAP_JSON = config.data_path / "Transcript" / "list_port_mcap_2015_2026.json"
UNIVERSE_JSON = config.data_path / "Transcript" / "list_portfolio_20151224.json"
CONFIG_DIR = config.research_path / "ca_strategy_poc" / "config"
OUTPUT_DIR = (
    config.research_path / "ca_strategy_poc" / "workspaces" / "full_run" / "output"
)
RESULT_DIR = config.research_path / "ca_strategy_poc" / "reports"
FRED_CACHE_DIR = config.data_path / "raw" / "fred"

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RISK_FREE_RATE_FALLBACK = 0.045  # annual fallback if FRED unavailable
RETURN_CAP = 0.50  # daily ±50%
START_DATE = date(2016, 1, 4)
END_DATE = date(2026, 2, 25)
TRADING_DAYS_PER_YEAR = 252

# Index column names in Bloomberg CSV
MSCI_KOKUSAI_COL = "MXKO INDEX"
MSCI_KOKUSAI_EW_COL = "MXKOEW INDEX"
MSCI_WORLD_COL = "MXWDJ INDEX"


In [2]:
rf = bt.fetch_dgs10_daily_rf()
price = bt.load_bloomberg_prices()
mcap = bt.load_bloomberg_mcap()

universe_data = bt.load_universe_data()


mapping = bt.build_bbg_ticker_map()


DGS10 loaded from cache: 2557 observations
Loading Bloomberg price data...
  Rows: 3169, Columns: 373
  Date range: 2015-12-24 ~ 2026-02-27
  Equity columns: 370
  Index columns: ['MXKO INDEX', 'MXKOEW INDEX', 'MXWDJ INDEX']
Loading Bloomberg market cap data...
  Rows: 720, Columns: 353
  Date range: 2014-12-31 ~ 2026-02-01


In [3]:
df_universe_data = pd.DataFrame.from_dict(universe_data, orient='index')
df_universe_data = df_universe_data[0].apply(pd.Series)
df_universe_data.index.name='sedol'
df_universe_data = df_universe_data.reset_index()
display(df_universe_data)


,sedol,Name,Country,GICS_Sector,GICS_Industry,MSCI_Mkt_Cap_USD_MM,KY,AK,Total,Target_Weight,LIST,date,Bloomberg_Ticker,FIGI
0,0003128,Aberdeen Asset Management PLC,UNITED KINGDOM,Financials,Diversified Financials,3997.131,,,,,LIST,2015-12-24T00:00:00.000,ADN LN Equity,BBG000BB8XR2
1,0059585,ARM Holdings plc,UNITED KINGDOM,Information Technology,Semiconductors & Semiconductor Equipment,21780.040,,,,,LIST,2015-12-24T00:00:00.000,ARM LN Equity,BBG000C21L21
2,0141192,Sky plc,UNITED KINGDOM,Consumer Discretionary,Media,16953.160,,,,,LIST,2015-12-24T00:00:00.000,SKY LN Equity,BBG000DXH4L2
3,0233527,Croda International Plc,UNITED KINGDOM,Materials,Materials,6171.147,,,,,LIST,2015-12-24T00:00:00.000,CRDA NQ Equity,BBG002LCGPG5
4,0237400,Diageo plc,UNITED KINGDOM,Consumer Staples,Food Beverage & Tobacco,69771.760,,,,,LIST,2015-12-24T00:00:00.000,DGE LN Equity,BBG000BS69D5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403,BYVY8G0,Alphabet Inc. Class A,UNITED STATES,Information Technology,Software & Services,222006.500,3,2,5,0.85,LIST,2015-12-24T00:00:00.000,GOOGL US Equity,BBG009S39JX6
404,BYX4D52,HP Inc.,UNITED STATES,Information Technology,Technology Hardware & Equipment,21189.250,,,,,LIST,2015-12-24T00:00:00.000,HPQ US Equity,BBG000KHWT55
405,BYXW419,Mr Price Group Limited,SOUTH AFRICA,Consumer Discretionary,Retailing,3200.691,,,,,LIST,2015-12-24T00:00:00.000,MRP SJ Equity,BBG000BTPJN9
406,BYY88Y7,Alphabet Inc. Class C,UNITED STATES,Information Technology,Software & Services,231656.800,3,2,5,0.85,LIST,2015-12-24T00:00:00.000,GOOG US Equity,BBG009S3NB30


In [4]:
display(price)


,005935 KS Equity,033780 KS Equity,035420 KS Equity,051900 KS Equity,090430 KS Equity,1044 HK Equity,11 HK Equity,1373183D US Equity,14 HK Equity,1482276D US Equity,...,WFC US Equity,WFM US Equity,WHL SJ Equity,WMT US Equity,WOW AU Equity,WTB LN Equity,XLNX US Equity,XOM US Equity,YUM US Equity,ZTS US Equity
Date,,,,,,,,,,,,,,,,,,,,,
2015-12-24,18.72403,95.33011,110.63422,884.04781,352.25116,9.08691,18.93851,146.32,4.13474,29.25,...,54.82,34.49,6.60390,20.2767,14.99834,56.93907,47.85,79.33,53.2100,48.13
2015-12-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-12-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-12-28,18.38786,94.07740,109.64293,881.76181,350.65213,9.00653,18.78516,146.35,4.13506,28.95,...,54.68,34.24,6.58622,20.2500,NaN,NaN,47.80,78.74,53.0949,47.90
2015-12-29,18.32168,90.49851,109.62272,902.42382,353.45645,9.11927,19.03128,147.94,4.11592,29.20,...,55.29,34.23,6.54597,20.5367,15.39899,56.95200,48.18,79.16,53.3826,48.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-23,94.27428,123.08992,178.01976,189.10270,109.09771,3.75742,NaN,NaN,2.94915,NaN,...,85.15,NaN,3.37505,125.8100,22.10173,35.37302,NaN,150.76,166.4500,125.79
2026-02-24,97.77525,121.02203,177.30004,187.36208,109.43333,3.75829,NaN,NaN,2.96061,NaN,...,84.57,NaN,3.41489,126.7500,22.28301,35.40364,NaN,149.26,165.9500,128.66
2026-02-25,99.76094,127.87347,177.71889,188.23480,110.27685,3.76480,NaN,NaN,2.93614,NaN,...,86.76,NaN,3.39088,125.7500,25.36500,35.36760,NaN,149.06,165.2300,128.95


In [5]:
returns = price.pct_change()
display(returns)


/var/folders/zk/q607hs0d7bq__dhkk2wj7sq80000gn/T/ipykernel_47511/3239021022.py:1: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = price.pct_change()


,005935 KS Equity,033780 KS Equity,035420 KS Equity,051900 KS Equity,090430 KS Equity,1044 HK Equity,11 HK Equity,1373183D US Equity,14 HK Equity,1482276D US Equity,...,WFC US Equity,WFM US Equity,WHL SJ Equity,WMT US Equity,WOW AU Equity,WTB LN Equity,XLNX US Equity,XOM US Equity,YUM US Equity,ZTS US Equity
Date,,,,,,,,,,,,,,,,,,,,,
2015-12-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-12-25,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2015-12-27,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2015-12-28,-0.017954,-0.013141,-0.008960,-0.002586,-0.004539,-0.008846,-0.008097,0.000205,0.000077,-0.010256,...,-0.002554,-0.007248,-0.002677,-0.001317,0.000000,0.000000,-0.001045,-0.007437,-0.002163,-0.004779
2015-12-29,-0.003599,-0.038042,-0.000184,0.023433,0.007997,0.012518,0.013102,0.010864,-0.004629,0.008636,...,0.011156,-0.000292,-0.006111,0.014158,0.026713,0.000227,0.007950,0.005334,0.005419,0.008977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-23,0.009921,-0.001433,0.010363,-0.001158,0.010197,0.009801,0.000000,0.000000,0.040081,0.000000,...,-0.040023,0.000000,0.002695,0.022929,0.000012,-0.039500,0.000000,0.023628,0.019102,-0.023218
2026-02-24,0.037136,-0.016800,-0.004043,-0.009205,0.003076,0.000232,0.000000,0.000000,0.003886,0.000000,...,-0.006812,0.000000,0.011804,0.007472,0.008202,0.000866,0.000000,-0.009950,-0.003004,0.022816
2026-02-25,0.020309,0.056613,0.002362,0.004658,0.007708,0.001732,0.000000,0.000000,-0.008265,0.000000,...,0.025896,0.000000,-0.007031,-0.007890,0.138311,-0.001018,0.000000,-0.001340,-0.004339,0.002254


In [6]:
display(mcap[['WTB LN Equity']].dropna())


,WTB LN Equity
Date,
2015-02-26,14796.0636
2015-08-27,13426.3267
2016-03-03,9765.8662
2016-09-01,10163.2999
2017-03-02,8588.1284
2017-08-31,8886.7666
2018-03-01,9631.0177
2018-08-30,9606.8083
2019-02-28,11575.5057


In [7]:
mcap_ffill = mcap.ffill()
initial_mcap = pd.DataFrame(mcap_ffill.loc['2015-12-31'])
display(initial_mcap)
display(initial_mcap.dropna())


,2015-12-31
033780 KS Equity,11194.6352
035420 KS Equity,16340.3313
051900 KS Equity,13090.5417
090430 KS Equity,20595.2606
1044 HK Equity,11504.9473
...,...
WTB LN Equity,13426.3267
XLNX US Equity,10774.9624
XOM US Equity,323960.2000
YUM US Equity,31080.0000


,2015-12-31
033780 KS Equity,11194.6352
035420 KS Equity,16340.3313
051900 KS Equity,13090.5417
090430 KS Equity,20595.2606
1044 HK Equity,11504.9473
...,...
WTB LN Equity,13426.3267
XLNX US Equity,10774.9624
XOM US Equity,323960.2000
YUM US Equity,31080.0000
